# 🛢️ Clasificación de Salud de Pozos — Well Health Risk
**Fuente de datos:** produccion_limpia.csv (local)

---

In [ ]:
# ==========================================
# 1. IMPORTAR LIBRERÍAS
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

In [ ]:
# ==========================================
# 2. CARGAR DATASET LOCAL
# ==========================================
df = pd.read_csv('produccion_limpia.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Filtrar solo pozos productores (los inyectores no tienen datos de producción)
df = df[df['Well_Type'].str.lower() == 'producer'].copy()

# Eliminar filas con NaN en las variables que usará el modelo
cols_modelo = ['Tubing_temp_F', 'Water_salinity_ppm', 'Scale_index',
               'Reservoir_pressure_psia', 'WaterCut_pct', 'Oil_rate_bbl_d',
               'Gas_rate_mscf_d', 'Sand_prod_lbs_d', 'Corrosion_inhibitor_ppm',
               'Paraffin_inhibitor_ppm', 'Lift_change_flag']
df = df.dropna(subset=cols_modelo).reset_index(drop=True)

print('Dataset cargado correctamente')
print(f'Registros: {df.shape[0]} | Pozos: {df["Well_ID"].nunique()}')
print(df.head())

In [ ]:
# ==========================
# REVISIÓN DEL DATASET
# ==========================
print('=' * 60)
print('DIMENSIONES DEL DATASET')
print('=' * 60)
print(df.shape)

print('\n' + '=' * 60)
print('COLUMNAS DEL DATASET')
print('=' * 60)
for col in df.columns:
    print(col)

print('\n' + '=' * 60)
print('TIPOS DE DATOS')
print('=' * 60)
print(df.dtypes)

In [ ]:
# ==========================
# VARIABLE TARGET
# ==========================
print(df['Lift_change_flag'].value_counts())
print('\nPorcentajes:')
print(df['Lift_change_flag'].value_counts(normalize=True) * 100)

In [ ]:
# ==========================================
# DISTRIBUCIÓN DEL TARGET
# ==========================================
plt.figure(figsize=(8, 5))
sns.countplot(x='Lift_change_flag', data=df)
plt.title('Distribución de Cambio de Levantamiento')
plt.xlabel('Lift Change Flag')
plt.ylabel('Cantidad')
plt.show()

In [ ]:
# ==========================================
# BOXPLOTS EXPLORATORIOS
# ==========================================
variables_box = ['Scale_index', 'Tubing_temp_F', 'Water_salinity_ppm',
                 'Oil_rate_bbl_d', 'Reservoir_pressure_psia']
titulos = ['Scale Index vs Cambio de Levantamiento',
           'Temperatura del Tubing vs Cambio',
           'Salinidad vs Cambio de Levantamiento',
           'Producción de Petróleo vs Cambio',
           'Presión del Reservorio vs Cambio']

for var, titulo in zip(variables_box, titulos):
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Lift_change_flag', y=var, data=df)
    plt.title(titulo)
    plt.xlabel('Cambio de levantamiento')
    plt.ylabel(var)
    plt.show()

In [ ]:
# ==========================================
# MATRIZ DE CORRELACIÓN
# ==========================================
variables = ['Oil_rate_bbl_d', 'Gas_rate_mscf_d', 'WaterCut_pct',
             'Reservoir_pressure_psia', 'Tubing_temp_F', 'Water_salinity_ppm',
             'Scale_index', 'Sand_prod_lbs_d', 'Corrosion_inhibitor_ppm',
             'Paraffin_inhibitor_ppm', 'Lift_change_flag']

corr = df[variables].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlación')
plt.show()

In [ ]:
# ==========================================
# MODELO 1: REGRESIÓN LOGÍSTICA CON LIFT_CHANGE_FLAG
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

# Variables del modelo
features = ['Tubing_temp_F', 'Water_salinity_ppm', 'Scale_index',
            'Reservoir_pressure_psia', 'WaterCut_pct', 'Oil_rate_bbl_d',
            'Gas_rate_mscf_d', 'Sand_prod_lbs_d', 'Corrosion_inhibitor_ppm',
            'Paraffin_inhibitor_ppm']
target = 'Lift_change_flag'

X = df[features]
y = df[target]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Balanceo con SMOTE
smote = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print('\nDistribución después de SMOTE:')
print(pd.Series(y_train_smote).value_counts())

# Modelo
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=2000)
model.fit(X_train_smote, y_train_smote)

# Predicciones
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Métricas
print('=' * 60)
print('REPORTE DE CLASIFICACIÓN')
print('=' * 60)
print(classification_report(y_test, y_pred))

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

# ROC AUC
auc = roc_auc_score(y_test, y_prob)
print(f'\nROC AUC: {round(auc, 4)}')

# Importancia de Variables
coeficientes = pd.DataFrame({
    'Variable': features,
    'Coeficiente': model.coef_[0]
})
coeficientes['Impacto_abs'] = abs(coeficientes['Coeficiente'])
coeficientes = coeficientes.sort_values(by='Impacto_abs', ascending=False)
print(coeficientes)

plt.figure(figsize=(10, 6))
sns.barplot(data=coeficientes, x='Coeficiente', y='Variable')
plt.title('Variables más influyentes en el riesgo de escalamiento')
plt.xlabel('Impacto en el modelo')
plt.ylabel('Variables')
plt.show()

In [ ]:
# ==========================================
# WELL HEALTH SCORE — TARGET MÁS ROBUSTO
# ==========================================
# Construcción del Health Score
df['health_score'] = 0

# 1. Riesgo químico de escalamiento
# Scale Index elevado
df['health_score'] += (df['Scale_index'] > df['Scale_index'].quantile(0.70)).astype(int)
# Alta Salinidad del Agua
df['health_score'] += (df['Water_salinity_ppm'] > df['Water_salinity_ppm'].quantile(0.70)).astype(int)
# Temperatura elevada del tubing
df['health_score'] += (df['Tubing_temp_F'] > df['Tubing_temp_F'].quantile(0.70)).astype(int)

# 2. Riesgo operacional
# Caída de presión del reservorio
df['health_score'] += (df['Reservoir_pressure_psia'] < df['Reservoir_pressure_psia'].quantile(0.30)).astype(int)
# Aumento del Water Cut
df['health_score'] += (df['WaterCut_pct'] > df['WaterCut_pct'].quantile(0.70)).astype(int)
# Disminución de producción de petróleo
df['health_score'] += (df['Oil_rate_bbl_d'] < df['Oil_rate_bbl_d'].quantile(0.30)).astype(int)

# TARGET FINAL: 3 o más señales = Riesgo Alto
df['Well_Health_Risk_v2'] = np.where(df['health_score'] >= 3, 1, 0)

# Distribución
print(df['Well_Health_Risk_v2'].value_counts())
print('\nPorcentajes:')
print(df['Well_Health_Risk_v2'].value_counts(normalize=True) * 100)

In [ ]:
# ==========================================
# MODELO FINAL — REGRESIÓN LOGÍSTICA CON WELL_HEALTH_RISK_V2
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

# Variables del modelo
features = ['Tubing_temp_F', 'Water_salinity_ppm', 'Scale_index',
            'Reservoir_pressure_psia', 'WaterCut_pct', 'Oil_rate_bbl_d',
            'Gas_rate_mscf_d', 'Sand_prod_lbs_d', 'Corrosion_inhibitor_ppm',
            'Paraffin_inhibitor_ppm']
target = 'Well_Health_Risk_v2'

X = df[features]
y = df[target]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print('Train:', X_train.shape)
print('Test:', X_test.shape)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print('\nDistribución después de SMOTE:')
print(pd.Series(y_train_smote).value_counts())

# Modelo
lr_model = LogisticRegression(random_state=42, max_iter=3000)
lr_model.fit(X_train_smote, y_train_smote)

# Predicciones
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Métricas
print('=' * 60)
print('REPORTE REGRESIÓN LOGÍSTICA')
print('=' * 60)
print(classification_report(y_test, y_pred_lr))

# Matriz
cm_lr = confusion_matrix(y_test, y_pred_lr)
print('\nMatriz de confusión:')
print(cm_lr)

# ROC AUC
auc_lr = roc_auc_score(y_test, y_prob_lr)
print(f'\nROC AUC: {auc_lr:.4f}')

In [ ]:
# ==========================================
# RED NEURONAL (MLPClassifier)
# ==========================================
from sklearn.neural_network import MLPClassifier

nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    learning_rate='adaptive',
    max_iter=500,
    random_state=42
)

# Entrenamiento
nn_model.fit(X_train_smote, y_train_smote)

# Predicciones
y_pred_nn = nn_model.predict(X_test_scaled)
y_prob_nn = nn_model.predict_proba(X_test_scaled)[:, 1]

# Reporte
print('=' * 60)
print('REPORTE RED NEURONAL')
print('=' * 60)
print(classification_report(y_test, y_pred_nn))

# Matriz
cm_nn = confusion_matrix(y_test, y_pred_nn)
print('\nMatriz de confusión:')
print(cm_nn)

# ROC
auc_nn = roc_auc_score(y_test, y_prob_nn)
print(f'\nROC AUC RED NEURONAL: {auc_nn:.4f}')

In [ ]:
# ==========================================
# GUARDAR MODELOS PARA EL DASHBOARD
# ==========================================
import joblib

# Guardar modelo neuronal
joblib.dump(nn_model, 'well_health_model.pkl')
# Guardar scaler
joblib.dump(scaler, 'scaler.pkl')
# Guardar features
joblib.dump(features, 'features.pkl')

print('✅ Modelos guardados correctamente:')
print('   - well_health_model.pkl')
print('   - scaler.pkl')
print('   - features.pkl')